# 01. Análisis exploratorio (EDA)


Objetivo: entender los datos antes de modelar. Miramos tamaño, tipos, valores perdidos, distribución de `price` y relación de cada variable con el precio. Las conclusiones de este notebook guían el preprocesado (02).

In [ ]:
import sys
from pathlib import Path

# Permite importar el paquete src/ desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from src.config import DATA_RAW, TARGET, CATEGORICAL, NUMERIC
sns.set_theme(style="whitegrid")

## 1. Carga de datos

In [ ]:
train = pd.read_csv(DATA_RAW / "train.csv")
test = pd.read_csv(DATA_RAW / "test.csv")
print("train:", train.shape, " test:", test.shape)
train.head()

In [ ]:
train.info()

In [ ]:
train.describe().T

## 2. Valores perdidos

El enunciado avisa de que puede haber valores perdidos. Comprobamos dónde están en train y en test.

In [ ]:
pd.DataFrame({"train": train.isna().sum(), "test": test.isna().sum()}).query("train > 0 or test > 0")

Solo `material` tiene huecos (alrededor del 3%). Miramos si el precio cambia cuando falta:

In [ ]:
train.groupby(train["material"].isna())[TARGET].describe()

## 3. Variable objetivo: `price`

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(train[TARGET], bins=60, ax=ax[0]).set_title("price")
sns.histplot(np.log(train[TARGET]), bins=60, ax=ax[1]).set_title("log(price)")
plt.show()
print("asimetría price:", round(train[TARGET].skew(), 2), " asimetría log(price):", round(np.log(train[TARGET]).skew(), 2))

`price` tiene una cola larga a la derecha (algunos productos superan los 1000 €). En escala logarítmica es casi simétrica. **Conclusión:** probar a entrenar sobre `log(price)`.

Como la métrica es RMSE, esos pocos precios muy altos pesan mucho en el error.

In [ ]:
train.nlargest(10, TARGET)

## 4. Variables numéricas

In [ ]:
train[NUMERIC].hist(bins=40, figsize=(14, 8))
plt.tight_layout(); plt.show()

In [ ]:
corr = train[NUMERIC + [TARGET]].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlaciones"); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.ravel(), NUMERIC):
    sns.scatterplot(data=train.sample(2000, random_state=0), x=col, y=TARGET, s=10, alpha=0.4, ax=ax)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

Observaciones:
- `weight_kg` es la variable más correlacionada con el precio (~0.57) y tiene cola larga: conviene su logaritmo.
- `shipping_cost` (~0.37) y `margin` (~0.24) suben el precio; `discount` e `is_promo` lo bajan.
- `reviews` también tiene cola larga.

## 5. Variables categóricas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), CATEGORICAL):
    orden = train.groupby(col)[TARGET].median().sort_values().index
    sns.boxplot(data=train, x=col, y=TARGET, order=orden, ax=ax, showfliers=False)
    ax.tick_params(axis="x", rotation=45)
axes.ravel()[-1].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
for col in CATEGORICAL:
    print(train.groupby(col, dropna=False)[TARGET].agg(["count", "mean"]).round(1), "\n")

In [ ]:
pd.crosstab(train["brand"], train["brand_tier"])

Observaciones:
- `category`: Electrónica es la más cara, Belleza la más barata.
- `brand_tier` tiene orden natural Budget < Media < Premium. Cada marca pertenece siempre al mismo tier (la tabla anterior), así que `brand` ya contiene la información de `brand_tier`.
- `season`, `channel`, `country` y `material` influyen poco.

## 6. ¿Train y test se parecen?

Si las distribuciones fueran distintas, la validación cruzada no sería fiable.

In [ ]:
comparacion = pd.DataFrame({"train": train[NUMERIC].mean(), "test": test[NUMERIC].mean()})
comparacion.round(3)

## 7. Conclusiones del EDA

1. 8000 filas de train, 2000 de test, 7 categóricas y 7 numéricas.
2. Solo `material` tiene valores perdidos (~3%): imputar con categoría `Desconocido`.
3. `price` es asimétrico: modelar `log(price)`.
4. `weight_kg` y `reviews` con cola larga: usar logaritmos.
5. `brand_tier` es ordinal.
6. Train y test tienen distribuciones parecidas, la CV de 5 folds es una buena estimación.